# Pricing a Box Spread

A box spread is long a combo at one strike $K_1$, short a combo at a higher strike $K_2$.

For each combo (long call, short put), Put-Call parity yields:

$$
C_{1} - P_{1} = e^{-rT} ( F - K_{1} )
$$
$$
C_{2} - P_{2} = e^{-rT} ( F - K_{2} )
$$

So the value for the box is:

$$
V = (C_{1} - P_{1}) - (C_{2} - P_{2}) =  e^{-rT} (K_2 - K_1)
$$

In the US, commonly traded boxes include those based on SPX options and those based on European ES options.

For the SPX case, we need only consider T+1 settlement as the expiration settlement is T+1 cash.

For the ES case, we would use T+0 settlement.

In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
# Reminder of standard settlement for US equities and options.
stock_settlement_days = 2
option_settlement_days = 1
box_rate = 2.5   # assume interest rate is 2.5%

## Example 1:   Calculate the value of a 1000 point box on SPX options expiring in a week.

Assumptions:  2.5% interest rate, Actual/360 convention, a week means 7 calendar days and no holidays intervene, so T+1 settlement days are also 7 days.

A "1000 point box" is one where the difference between the upper and lower strikes is $1000.

For each overnight, the rate factor is $1 - r/360$ (given the Actual/360 convention.)

Let's say the trade date is Wednesday, and the options expire the following Wednesday.  The options settle T+1, i.e., Thursday, so this is the first overnight carry.  All options expire the following Wednesday, and are either exercised or abandoned.  Those position changes don't settle till the next day, Thursday, so carry continues through that Wednesday overnight. On Thursday there is no carry as all settled positions are zero at that point.

In [ ]:
strike_difference = 1000
rate_factor = (1 - box_rate/100/360) ** 7
box_price = rate_factor * strike_difference
print(f"box_price: {box_price}")

## Example 2:  Calculate a box when there is a term structure of rates.

Suppose that today is Tuesday, Dec 7 2021 and that rates are currently at 1.5%.
Additionally, suppose that at the end of the Dec 14-15 FOMC meeting the Fed is widely expected to raise rates 50 bp.  
Price the 1000 point SPX box expiring Friday Dec 17, 2021.

In [ ]:
import datetime
import pandas as pd
import numpy as np
from pandas.tseries.offsets import CustomBusinessDay
from pandas.tseries.holiday import USFederalHolidayCalendar

holiday_calendar = USFederalHolidayCalendar()
cbday = CustomBusinessDay(calendar=USFederalHolidayCalendar())

def get_option_settlement_date(dt):
    return (dt + option_settlement_days*cbday).date()

In [ ]:
# The dates we care about
trade_date = datetime.date(2021, 12, 7)
expiration_date = datetime.date(2021, 12, 17)
fomc_announcement_date = datetime.date(2021, 12, 15)

trade_settle = get_option_settlement_date(trade_date)
expiration_settle = get_option_settlement_date(expiration_date)

In [ ]:
# We'll construct a dataframe with information that is unnecessary, but helpful for visualizing
dates = pd.date_range(trade_date, expiration_settle)
weekday = [d.date().strftime("%a") for d in dates]
df = pd.DataFrame(data={'dt': dates, 'weekday': weekday}, columns=['dt', 'weekday'])

In [ ]:
df['is_trade_date'] = np.where(df['dt'] == pd.Timestamp(trade_date), 1, 0)
df['is_expiration_date'] = np.where(df['dt'] == pd.Timestamp(expiration_date), 1, 0)
df['carry_flag'] = [d.date() >= trade_settle and d.date() < expiration_settle for d in dates]

In [ ]:
# a Fed rate change takes effect on the business day following the announcement
initial_rate = 0.015  # pre-FOMC rate
raised_rate = 0.020  # post-FOMC rate
df['rate'] = np.where(df['dt'] > pd.Timestamp(fomc_announcement_date), raised_rate, initial_rate)
df

In [ ]:
# array of Actual/360 rates for each carry day
r = df.query('carry_flag')['rate'].values
display(r)

In [ ]:
rate_factor_example2 = np.prod(1.0 - r/360)
box_price = strike_difference * rate_factor_example2
print(f"box_price is {box_price:.2f}")

Note that in this case the settlement affects not only the number of days but also offset which days we use from the rate curve.

## Example 3:   A real life example from the SPX floor

From slack on 2022-07-18, SPX floor trader reports:

"SPX oct 1000 pt box paper buys 1k @ 993.25"

If we model the box rate as Fed funds plus a constant offset, what offset is implied by this trade?

In [ ]:
frc = pd.read_csv("FedFundsRateCurve_20220718.csv", parse_dates=["dtStr"])
display(frc)

trade_settle_date = get_option_settlement_date(datetime.date(2022, 7, 18))
expiration_settle_date = get_option_settlement_date(datetime.date(2022, 10, 21))
print(f"trade_settle_date: {trade_settle_date} expiration_settle_date: {expiration_settle_date}")

# There are 3 rate intervals (all inclusive of endpoints):
#  2022-07-19 to 2022-07-27 
#  2022-07-28 to 2022-09-21
#  2022-09-22 to 2022-10-23

In [ ]:
interval_endpoints = [trade_settle_date] + [x.date() for x in frc['dtStr'][1:3]] + [expiration_settle_date]
interval_endpoints

In [ ]:
day_count = np.array([x.days for x in np.diff(np.array(interval_endpoints))])
fed_rates = np.array(frc['rate'][0:3])
print(f"day_count: {day_count}")
print(f"fed_rates: {fed_rates}")

In [ ]:
import scipy
from scipy.optimize import brentq

def box_price(offset):
    days_in_year = 360.
    rates_to_use = fed_rates + offset
    log_factor = np.dot(day_count, np.log(1.0 + rates_to_use/100/days_in_year))
    return 1000 * np.exp(-log_factor)

traded_price = 993.25

result = scipy.optimize.brentq(lambda x: box_price(x) - traded_price, 0.0, 0.50)
print(f"offset result: {result:.3f} % or {100*result:.1f} bp")